In [ ]:
# Install dependencies and import functions
!pip install tensorflow opencv-python matplotlib

import tensorflow as tf
import os
from matplotlib import pyplot as plt
import cv2
import imghdr
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Input, Activation
import sys
from tensorflow.keras.metrics import Precision, Recall, BinaryAccuracy

In [ ]:
# 1. SET UP AND LOAD DATA

# Avoid OOM errors by setting GPU memory consumption growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu,True)

#Load and clean up data
data_dir = 'Data_CNN' #update path as needed

#remove images with incompatable extensions (i.e., not common image extensions)
image_exts = ['jpeg','JPEG','jpg','JPG','bmp','BMP','png','PNG']

for image_class in os.listdir(data_dir):
    for image in os.listdir(os.path.join(data_dir, image_class)):
        image_path = os.path.join(data_dir, image_class, image)
        try:
            img = cv2.imread(image_path)
            tip = imghdr.what(image_path)
            if tip not in image_exts:
                print('Image not in ext list {}'.format(image_path))
                os.remove(image_path)
        except Exception as e:
            print('Issue with image {}'.format(image_path))
            # os.remove(image_path)

#load data
data = tf.keras.utils.image_dataset_from_directory(
    'Data_CNN',
    labels='inferred',
    label_mode='categorical', #ensures labels are one-hot encoded
    image_size=(256,256),
    batch_size=64,
    shuffle=True
)
#this is where we load out dataset
#builds image dataset for you on the fly (pre-procesing, labeling, resize image, batch by 32, etc)
#this builds the data pipeline

data_iterator = data.as_numpy_iterator() #allows us to access generator from our data pipeline (keras generator)

batch = data_iterator.next() #get another batch from the iterator when you re-run this code

In [ ]:
# 2. DATA PRE-PROCESSING

#scale data
data = data.map(lambda x,y: (x/255, y))
# we devide the image shape (0-255) by 255 to scale from 0-1 (images are imported as 256x256)
# x = images
# y = target variable

data.as_numpy_iterator().next() #grabs next batch

# split data
train_size = round(len(data)*.7) #round up to 19 batches out of 27 total
val_size = round(len(data)*.15) #round down to 4 batches out of 27 total
test_size = round(len(data)*.15) #round up to 4 batches out of 27 total

#establish train, test, and validation partitions
train = data.take(train_size) #allocates 19 batches to train dataset
val = data.skip(train_size).take(val_size) #skip batches we already allocated to training partition and allocate 4 batches to validation 
                                           #partition
test = data.skip(train_size+val_size).take(test_size) #skip batches we already allocated to training and validation aprtitions and allocate 
                                                      #4 batches to test partition

In [ ]:
# 3. BUILD DEEP LEARNING MODEL

#model: 3 convolution blocks, flatten layer, 2 dense layers

model = Sequential([
    Input(shape=(256, 256, 3)),  # Define the input shape 
    
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(),
    
    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(),
    
    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(),
    
    # Flattening the convolved images to feed into the dense layers
    Flatten(),
    
    # Dense Layer
    Dense(128, activation='relu'),
    
    # Output Layer with 25 units (one for each class)
    Dense(25, activation='softmax'),
])

#compile neural network
model.compile(optimizer='adam', #adam = optimizer
              loss='categorical_crossentropy', #loss = categorical cross entropy because multi-class model
              metrics=['accuracy']) #track accuracy metrics, which tells us how well our model is classifying


# Model summary
model.summary()


In [ ]:
# 4. TRAIN NEURAL NETWORK

logdir = 'logs' #log directory
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir) #create callback to log model or save at particluar check point

#function to supporess stderr
def suppress_stderr():
    sys.stderr = open(os.devnull, 'w')

#function to restore stderr
def restore_stderr():
    sys.stderr = sys.__stderr__

#train the model
hist = model.fit(
    train, 
    epochs=30, 
    validation_data=val, 
    callbacks=[tensorboard_callback]
)

# To train
#hist = model.fit(x_train, y_train, epochs=30, batch_size=32, validation_data=(x_val, y_val))

#restore stderr after training
restore_stderr()

#gather history (accuracy and loss metrics from training and val)
hist.history

In [ ]:
# 5. EVALUATE MODEL PERFORMANCE

#visualize loss
fig = plt.figure()
plt.plot(hist.history['loss'], color='teal', label='loss')
plt.plot(hist.history['val_loss'], color='orange', label='val_loss')
plt.title('Loss', fontsize=20)
plt.legend(loc="upper left")
plt.show()

#visualize accuracy
fig = plt.figure()
plt.plot(hist.history['accuracy'], color='teal', label='accuracy')
plt.plot(hist.history['val_accuracy'], color='orange', label='val_accuracy')
plt.title('Accuracy', fontsize=20)
plt.legend(loc='upper left')
plt.show

In [ ]:
# 6. TEST MODEL

#evaluate performance on testing batch (that model has never seen before)
pre = Precision()
re = Recall()
acc = BinaryAccuracy()

for batch in test.as_numpy_iterator(): #loop through test batches and update the values for pre, re, acc
    X, y = batch #set of images
    yhat = model.predict(X) #pass image through model and make predictions (outputs value sbetween 0-1)
    pre.update_state(y, yhat) #update metrics (pass through y true and yhat value)
    re.update_state(y, yhat)
    acc.update_state(y, yhat)

#print metrics
print(f'Precision: {pre.result().numpy()}, Recall: {re.result().numpy()}, Accuracy: {acc.result().numpy()}')

In [ ]:
#7. SAVE THE MODEL

#use the load_model function to be able to load our model (so we can save it, add it to API, edge device, etc)

#save model
model.save(os.path.join('models','CNN2.h5')) #change path and saved model name as desired

#load model
new_model = load_model(os.path.join('models','CNN2.h5')) #change path and model name if needed
#this loads the model